In [ ]:
import sys
!{sys.executable} -m pip install -q playwright pandas openpyxl
!playwright install chromium
!playwright install-deps chromium   # ← this is the critical fix

In [ ]:
# ============================================================
# CELL 2 — OLX Scraper  (run after Cell 1 completes)
# Scrapes 3 pages of listings, faster but human-like
# ============================================================

import asyncio
import random
import re
from datetime import datetime

import pandas as pd
from playwright.async_api import async_playwright


# =========================================================
# CONFIG
# =========================================================

BASE_URL   = "https://www.olx.uz/nedvizhimost/kvartiry/prodazha/?currency=UZS"
MAX_PAGES  = 5          # number of listing pages to collect links from

OUTPUT_CSV  = "olx_scraped_v2.csv"
OUTPUT_XLSX = "olx_scraped_v2.xlsx"


# =========================================================
# TIMING PROFILES
# Human-like but faster: shorter waits, lighter scrolls,
# occasional micro-pauses to mimic reading behaviour.
# =========================================================

# Wait after loading a listing page (ms)  — was 4000–8000
AD_WAIT_MS   = (1800, 3500)

# Wait between ads (seconds)             — was 8–16 s
BETWEEN_ADS  = (2.0, 4.0)

# Wait between list pages (seconds)
BETWEEN_LIST = (4.0, 8.0)

# Long break every N ads (seconds)       — was 30–60 s
LONG_BREAK_EVERY = 15
LONG_BREAK_SECS  = (10, 15)

# Scroll: fewer passes, shorter distance — was 3–5 passes of 800–2000 px
SCROLL_PASSES  = (2, 3)
SCROLL_DIST_PX = (500, 1200)
SCROLL_PAUSE   = (0.4, 0.9)


# =========================================================
# HELPERS
# =========================================================

def clean(text):
    if not text:
        return None
    return re.sub(r"\s+", " ", text).strip()

def extract_number(text):
    if not text:
        return None
    text = text.replace("\u00a0", "").replace(" ", "")
    m = re.search(r"(\d+)", text)
    return int(m.group(1)) if m else None

def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

async def short_delay(a, b):
    await asyncio.sleep(random.uniform(a, b))

async def human_scroll(page, fast=False):
    passes = 1 if fast else random.randint(*SCROLL_PASSES)
    for _ in range(passes):
        dist = random.randint(*SCROLL_DIST_PX)
        await page.mouse.wheel(0, dist)
        await asyncio.sleep(random.uniform(*SCROLL_PAUSE))
        # occasionally pause mid-scroll as if reading
        if not fast and random.random() < 0.3:
            await asyncio.sleep(random.uniform(0.5, 1.2))


# =========================================================
# COLLECT LISTING LINKS — multi-page
# =========================================================

def page_url(base: str, page_num: int) -> str:
    """Append &page=N to the base URL."""
    sep = "&" if "?" in base else "?"
    return f"{base}{sep}page={page_num}" if page_num > 1 else base

async def get_all_links(page, max_pages: int) -> list:
    all_links = set()

    for pg in range(1, max_pages + 1):
        url = page_url(BASE_URL, pg)
        print(f"\n── LIST PAGE {pg}/{max_pages}: {url}")

        await page.goto(url, wait_until="networkidle", timeout=60000)
        await page.wait_for_timeout(random.randint(2000, 4000))
        await human_scroll(page)

        hrefs = await page.locator("a").evaluate_all(
            "elements => elements.map(e => e.href)"
        )
        before = len(all_links)
        for href in hrefs:
            if href and "/d/obyavlenie/" in href:
                all_links.add(href.split("?")[0])

        print(f"   +{len(all_links) - before} new  (total {len(all_links)})")

        if pg < max_pages:
            await short_delay(*BETWEEN_LIST)

    return list(all_links)


# =========================================================
# STRUCTURED ATTRIBUTE LIST (data-nx-name="ListContainer")
# =========================================================

async def get_list_container_attrs(page):
    attrs = {}
    try:
        container = page.locator('[data-nx-name="ListContainer"]')
        if await container.count() == 0:
            return attrs
        rows = container.locator("li, p")
        count = await rows.count()
        for i in range(count):
            row_text = clean(await rows.nth(i).inner_text())
            if not row_text:
                continue
            if ":" in row_text:
                key, _, val = row_text.partition(":")
                attrs[clean(key)] = clean(val)
            else:
                parts = row_text.split(None, 1)
                if len(parts) == 2:
                    attrs[clean(parts[0])] = clean(parts[1])
    except Exception as e:
        print(f"  [attrs] {e}")
    return attrs


# =========================================================
# SCRAPE ONE AD
# =========================================================

async def scrape_ad(page, url):
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(random.randint(*AD_WAIT_MS))
        await human_scroll(page)

        page_title = await page.title()
        if any(x in page_title.lower() for x in ["403", "access denied", "captcha", "just a moment"]):
            print(f"  BLOCKED: {url}")
            return None

        # ── 1. LISTING ID ─────────────────────────────────────────
        listing_id = None
        try:
            label2 = page.locator('[data-nx-name="Label2"]')
            if await label2.count() > 0:
                raw = clean(await label2.first.inner_text()) or ""
                m = re.search(r"(\d{5,})", raw)
                if m:
                    listing_id = m.group(1)
        except Exception:
            pass

        if not listing_id:
            try:
                body = clean(await page.text_content("body")) or ""
                m = re.search(r"\bID[:\s#]*(\d{5,})", body, re.I)
                if m:
                    listing_id = m.group(1)
            except Exception:
                pass

        if not listing_id:
            m = re.search(r"-(ID[A-Za-z0-9]+)\.html", url)
            if m:
                listing_id = m.group(1)

        # ── 2. TITLE ──────────────────────────────────────────────
        title = None
        try:
            h1 = page.locator("h1")
            if await h1.count() > 0:
                title = clean(await h1.first.inner_text())
        except Exception:
            pass
        if not title:
            title = clean(page_title.split(":")[0])

        # ── 3. PRICE & CURRENCY ───────────────────────────────────
        price = None
        currency = None
        try:
            price_loc = page.locator('[data-testid="ad-price-container"]')
            price_text = ""
            if await price_loc.count() > 0:
                price_text = clean(await price_loc.first.inner_text()) or ""
            else:
                price_text = clean(await page.text_content("body")) or ""
            price = extract_number(price_text)
            low = price_text.lower()
            if "$" in price_text or "у.е" in low or "usd" in low:
                currency = "USD"
            elif "сум" in low or "uzs" in low or "sum" in low:
                currency = "UZS"
        except Exception:
            pass

        # ── 4. STRUCTURED ATTRS (ListContainer) ──────────────────
        attrs = await get_list_container_attrs(page)

        def pick(keys):
            for k in keys:
                if attrs.get(k):
                    return attrs[k]
            return None

        area_raw   = pick(["Общая площадь", "Umumiy maydoni", "Умумий майдони"])
        rooms_raw  = pick(["Количество комнат", "Xonalar soni", "Xona soni"])
        market_raw = pick(["Тип жилья", "Uy-joy turi"])
        stair_raw  = pick(["Этаж", "Qavat"])

        area        = clean(area_raw)
        num_rooms   = extract_number(rooms_raw)
        market_type = clean(market_raw)
        stair       = clean(stair_raw)

        # Body-text regex fallback
        if not all([area, num_rooms, stair]):
            try:
                bt = clean(await page.text_content("body")) or ""
                if not area:
                    m = re.search(r"Общая площадь[:\s]*([^\n\r]{1,30}?)(?=\s*(?:Этаж|Количество|Тип|$))", bt, re.I)
                    if m:
                        val = clean(m.group(1))
                        if val and re.search(r"\d", val) and len(val) < 30:
                            area = val
                if not num_rooms:
                    m = re.search(r"Количество комнат[:\s]*(\d+)", bt, re.I)
                    if m:
                        num_rooms = int(m.group(1))
                if not market_type:
                    m = re.search(r"Тип жилья[:\s]*([^\n\r]{1,60}?)(?=\s*(?:Этаж|Количество|Общая|$))", bt, re.I)
                    if m:
                        val = clean(m.group(1))
                        if val and len(val) < 60:
                            market_type = val
                if not stair:
                    m = re.search(r"\bЭтаж[:\s]*([^\n\r]{1,30}?)(?=\s*(?:Количество|Общая|Тип|$))", bt, re.I)
                    if m:
                        val = clean(m.group(1))
                        if val and len(val) < 30:
                            stair = val
            except Exception:
                pass

        # ── 5. VIEWS ──────────────────────────────────────────────
        views = None
        try:
            views_loc = page.locator('[data-testid="page-view-counter"]')
            if await views_loc.count() > 0:
                views = extract_number(await views_loc.first.inner_text())
        except Exception:
            pass

        # ── 6. POSTED DATE ────────────────────────────────────────
        posted_date = None
        try:
            date_loc = page.locator('[data-cy="ad-posted-at"], [data-testid="ad-posted-at"]')
            if await date_loc.count() > 0:
                posted_date = clean(await date_loc.first.inner_text())
            else:
                bt = clean(await page.text_content("body")) or ""
                m = re.search(r"Опубликовано[:\s]*([^\n\r]{3,40})", bt, re.I)
                if not m:
                    m = re.search(r"Дата публикации[:\s]*([^\n\r]{3,40})", bt, re.I)
                if m:
                    posted_date = clean(m.group(1))
        except Exception:
            pass

        # ── 7. NEGOTIATION ────────────────────────────────────────
        negotiation = False
        try:
            p4 = page.locator('[data-nx-name="P4"]')
            if await p4.count() > 0:
                p4_text = (clean(await p4.first.inner_text()) or "").lower()
                if any(x in p4_text for x in ["договорная", "negotiable", "kelishiladi"]):
                    negotiation = True
            if not negotiation:
                bt = (clean(await page.text_content("body")) or "").lower()
                if "договорная" in bt or "negotiable" in bt:
                    negotiation = True
        except Exception:
            pass

        # ── 8. SELLER NAME ────────────────────────────────────────
        seller = None
        try:
            seller_loc = page.locator('[data-testid="user-profile-user-name"]')
            if await seller_loc.count() > 0:
                seller = clean(await seller_loc.first.inner_text())
        except Exception:
            pass

        # ── 9. LOCATION — proven working approach ─────────────────
        location = None
        try:
            texts = await page.locator("p, span").all_inner_texts()
            for text in texts:
                text = clean(text)
                if not text:
                    continue
                low = text.lower()
                if any(x in low for x in ["район", "ташкент", "toshkent", "область"]):
                    if len(text) < 120:
                        location = text
                        break
        except Exception:
            pass

        # ── 10. SELLER JOINED ─────────────────────────────────────
        seller_joined = None
        try:
            ms_loc = page.locator('[data-testid="member-since"]')
            if await ms_loc.count() > 0:
                seller_joined = clean(await ms_loc.first.inner_text())
        except Exception:
            pass
        if not seller_joined:
            try:
                texts = await page.locator("span, p").all_inner_texts()
                for text in texts:
                    text = clean(text)
                    if text and "на olx с" in text.lower():
                        seller_joined = text
                        break
            except Exception:
                pass

        # ── 11. DESCRIPTION ───────────────────────────────────────
        description = None
        try:
            desc_loc = page.locator('[data-testid="ad_description"]')
            if await desc_loc.count() > 0:
                description = clean(await desc_loc.first.inner_text())
        except Exception:
            pass

        # ── 12. DESCRIPTION FALLBACK ──────────────────────────────
        if description:
            desc_low = description.lower()
            if not area:
                m = re.search(r"(\d+[\.,]?\d*)\s*м[²2]", description, re.I)
                if m:
                    area = m.group(1) + " м²"
            if not num_rooms:
                m = re.search(r"(\d+)[\s-]*комнат", description, re.I)
                if not m:
                    m = re.search(r"комнат[:\s]*(\d+)", description, re.I)
                if m:
                    num_rooms = int(m.group(1))
            if not stair:
                m = re.search(r"этаж[:\s]*([\d\s/\-]+(?:из\s*\d+)?)", description, re.I)
                if m:
                    stair = clean(m.group(1))
            if not market_type:
                if any(x in desc_low for x in ["вторичный", "вторичка"]):
                    market_type = "Вторичный рынок"
                elif any(x in desc_low for x in ["новостройка", "первичный"]):
                    market_type = "Новостройка"

        return {
            "Listing ID":    listing_id,
            "Title":         title,
            "Price":         price,
            "Currency":      currency,
            "Area":          area,
            "Num Rooms":     num_rooms,
            "Market Type":   market_type,
            "Views":         views,
            "Stair":         stair,
            "Posted Date":   posted_date,
            "Scraped Date":  now(),
            "Negotiation":   negotiation,
            "Seller":        seller,
            "Location":      location,
            "Seller Joined": seller_joined,
            "Description":   description,
            "URL":           url,
        }

    except Exception as e:
        print(f"  FAILED: {e}")
        return None


# =========================================================
# MAIN
# =========================================================

async def main():
    all_data = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            slow_mo=80,                        # was 100 — slightly faster
            args=["--no-sandbox", "--disable-dev-shm-usage"],
        )
        context = await browser.new_context(
            viewport={"width": 1400, "height": 900},
            locale="ru-RU",
            timezone_id="Asia/Tashkent",
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
        )
        page = await context.new_page()

        await page.add_init_script("""
            Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
            window.chrome = { runtime: {} };
            Object.defineProperty(navigator, 'plugins',   { get: () => [1,2,3,4,5] });
            Object.defineProperty(navigator, 'languages', { get: () => ['ru-RU','ru','en-US'] });
        """)

        # Quick warm-up — shorter than before (was 5–10 s)
        print("WARMING SESSION...")
        await page.goto("https://www.olx.uz/", wait_until="domcontentloaded")
        await short_delay(3, 5)
        await human_scroll(page, fast=True)

        # ── Collect links from all 3 pages ────────────────────────
        all_links = await get_all_links(page, MAX_PAGES)
        all_links = list(set(all_links))
        print(f"\nTOTAL UNIQUE LINKS: {len(all_links)}")
        print("Starting ad scrape...\n")

        # ── Scrape each ad ────────────────────────────────────────
        for idx, link in enumerate(all_links, start=1):
            print(f"[{idx}/{len(all_links)}] {link.split('/')[-1]}")
            data = await scrape_ad(page, link)
            if data:
                all_data.append(data)
                print(
                    f"  ✓ ID={data['Listing ID']} | "
                    f"loc={data['Location']} | "
                    f"{(data['Title'] or '')[:45]}"
                )
            else:
                print(f"  ✗ skipped")

            # Short gap between ads
            await short_delay(*BETWEEN_ADS)

            # Occasional longer pause — mimics user taking a break
            if idx % LONG_BREAK_EVERY == 0:
                secs = random.uniform(*LONG_BREAK_SECS)
                print(f"\n  ── pause {secs:.0f}s (every {LONG_BREAK_EVERY} ads) ──\n")
                await asyncio.sleep(secs)

        await browser.close()

    # ── Save ─────────────────────────────────────────────────────
    df = pd.DataFrame(all_data)
    col_order = [
        "Listing ID", "Title", "Price", "Currency",
        "Area", "Num Rooms", "Market Type", "Views",
        "Stair", "Posted Date", "Scraped Date", "Negotiation",
        "Seller", "Location", "Seller Joined",
        "Description", "URL",
    ]
    df = df[[c for c in col_order if c in df.columns]]
    df.to_csv(OUTPUT_CSV,  index=False, encoding="utf-8-sig")
    df.to_excel(OUTPUT_XLSX, index=False)

    print(f"\n{'='*50}")
    print(f"DONE  —  {len(df)} listings saved")
    print(f"  → {OUTPUT_XLSX}")
    print(f"  → {OUTPUT_CSV}")
    print(f"{'='*50}")


await main()